# Carga de archivos Gpkg a Athena

Este script se encarga de:

- Leer los archivos .gpkg guardados en Athena (o de forma local) resultado de la transformacion en 2_AP_ShpToGpkh.ipynb
- Preprocesado de ficheros para concordar con el formato aceptado por la tabla de Athena
- Cargar los archivos transformados a Athena

<a id='contents' />

## Tabla de contenidos

1. [Carga de librerías](#loading)
2. [Definicion de metodos](#methods)
3. [Definicion de variables](#variables)
4. [Scripts principales](#scripts)

<a id='loading' />

## Carga de librerías
[(back to top)](#contents)

In [5]:
%load_ext autoreload
%autoreload 2
%run config_notebook.py

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Directorio "src" adicionado para cargar modulos


ModuleNotFoundError: No module named 'sagemaker'

In [6]:
import boto3
import geopandas as gpd
import pandas as pd
import awswrangler as wr
import shapely.wkt
import numpy as np
import json
from decimal import Decimal

ModuleNotFoundError: No module named 'geopandas'

<a id='methods' />

## Definicion de metodos
[(back to top)](#contents)

In [ ]:
def set_shapes_bucket(ssm, aws_account_env):
    env_name = "prod" if aws_account_env == "produccion" else "dev"
    if (aws_account_env != "hel001"):
        return f"nutresa-datalake-{env_name}-raw"
    else:
        return ssm.get_parameter(Name='/ml/s3/foundation/feature_store_bucket_name')['Parameter']['Value']
    
def set_monitoring_name(ssm, aws_account_env):
    return ssm.get_parameter(Name='/ml/s3/foundation/monitoring_bucket_name')['Parameter']['Value']

def set_dane_folder_name(env_name, is_manzana):
    if (env_name != "hel001"):
        path = "engineering/datos-no-estructurados_tranversales/mapas_evp/ftp/mgn_municipio_politico/"
        if is_manzana:
            path = "engineering/datos-no-estructurados_tranversales/mapas_evp/ftp/manzana/"
    else:
        path = "Dane_shapes/MGN2020_MPIO_POLITICO/"
        if is_manzana:
            path = "Dane_shapes/MANZANA/"
    return path

def rename_columns(shapedf: gpd.GeoDataFrame):
    columnsrename1 = {
        'tp3_a_ri': 'TP3A_RI', 
        'tp51_postgr': 'TP51POSTGR',
        'tp3_b_tcn': 'TP3B_TCN',
        'tp51_secund': 'TP51SECUND',
        'tp51_primar': 'TP51PRIMAR',
        'tp51_superi': 'TP51SUPERI'}
    shapedf:gpd.GeoDataFrame = shapedf.rename(columns=columnsrename1)
    column_names = dict(zip(shapedf.columns, map(str.upper, shapedf.columns)))
    shapedf.rename(columns=column_names, inplace=True)
    columnsrename2 = {
        'SHAPE_LENG': 'Shape_Leng', 
        'SHAPE_AREA': 'Shape_Area',
        'GEOMETRY': 'geometry'}
    shapedf = shapedf.rename(columns=columnsrename2)
    return shapedf

def parse_columns_dtypes(shapedf: gpd.GeoDataFrame):
    with open('schema.json', 'r') as f:
        dane_schema = json.load(f)    
    shapedf = shapedf.astype(dane_schema)
    
    columns = shapedf.columns.to_list()
    for col in columns:
        if shapedf.dtypes[col] == 'float64':
            shapedf[col] = shapedf[col].apply(lambda x: np.float64(x))
    return shapedf

def count_decimals(x):
    """Returns the number of decimal places in a float."""
    if isinstance(x, float):
        return (Decimal(str(x)).as_tuple().exponent * -1)
    else:
        return 0

<a id='variables' />

## Definicion de variables
[(back to top)](#contents)

Codigo de registro de Dane a validar

In [ ]:
daneCode = "11001"

Cliente SSM

In [ ]:
ssm = boto3.client('ssm', region_name='us-east-1')
aws_account_env = ssm.get_parameter(Name='/ml/misc/foundation/env_name')['Parameter']['Value']
featurestore_bucket = set_shapes_bucket(ssm, aws_account_env)
dane_path = set_dane_folder_name(ssm, aws_account_env)
shape_path = f"s3://{featurestore_bucket}/{dane_path}{daneCode}.shp"
gpkg_path = f"geoPackages/{daneCode}.gpkg"

<a id='scripts' />

## Scripts principales
[(back to top)](#contents)

### Lectura de Geopackage

In [ ]:
geopackage = gpd.read_file(gpkg_path)

Creacion de GeoDataFrame y cambio de CRS

In [ ]:
projection_code = "EPSG:3116"
geopackage = gpd.GeoDataFrame(geopackage, crs=projection_code)
geopackage = geopackage.to_crs(projection_code)
geopackage = geopackage.sort_values(by='COD_DANE_A').reset_index(drop=True)

In [ ]:
geopackage

### Lectura de Geopackage en Tabla

In [ ]:
query = f"""
SELECT * FROM "tb_ml_helados_gpkgs" WHERE mpio_cdpmp LIKE '{daneCode}'
"""
ruta_s3 = "s3://datalab-helados-athena-resultados/athena/"
region = 'us-east-1'
database_name = 'db_machine_learning_helados_lab'
projection_code = "EPSG:3116"

geopackage2 = wr.athena.read_sql_query(
                query,
                database=database_name,
                ctas_approach=False,
                s3_output=ruta_s3,
                boto3_session=boto3.session.Session(region_name=region)
                )
geopackage2['geometry'] = geopackage2['geometry'].apply(lambda x: shapely.wkt.loads(x))
geopackage2 = rename_columns(geopackage2)
geopackage2 = parse_columns_dtypes(geopackage2)
geopackage2 = gpd.GeoDataFrame(geopackage2, crs="EPSG:4326")
geopackage2 = geopackage2.to_crs(projection_code)
geopackage2 = geopackage2.replace({pd.NA: None})
geopackage2 = geopackage2.sort_values(by='COD_DANE_A').reset_index(drop=True)
geopackage2.crs

In [ ]:
geopackage2

### Comparacion de Geodataframes

Validacion de CRS's

In [ ]:
if geopackage.crs != geopackage2.crs:
    print("The two GeoDataFrames have different coordinate reference systems and cannot be compared.")
else:
    print("The two GeoDataFrames have the same coordinate reference systems.")

Validacion de dtypes

In [ ]:
types = geopackage.dtypes
types2 = geopackage2.dtypes
types == types2

Validacion de equivalencia general

In [ ]:
geopackage.equals(geopackage2)

Validacion de diferencias generales entre columnas de Geodataframes

In [ ]:
# Transform column data to array in order to ease the comparisson
columns = geopackage.columns
difference_columns = []
for col in columns.to_list():
    print(col)
    print("\n")
    print(f"geopackage {col} dtype: {geopackage.dtypes[col]}")
    print(f"geopackage2 {col} dtype: {geopackage2.dtypes[col]}")
    print(f"Initial data on geopackage: {type(geopackage[col][0])}")
    print(geopackage[col][0])
    print(f"Initial data on geopackage2: {type(geopackage2[col][0])}")
    print(geopackage2[col][0])
    print("\n")
    try:
        # Merge the two GeoDataFrames based on the column of interest, and keep track of the differences
        arr1 = np.array(geopackage[col])
        arr2 = np.array(geopackage2[col])
        diff = np.setdiff1d(arr1, arr2)

        # Print the differences
        if len(diff) == 0:
            print("No differences between the two GeoDataFrames")
        else:
            print(f"Differences between the two GeoDataFrames: {len(diff)}/{len(arr1)}")
            print(diff)
    except Exception as e:
        print(f"Exception: {e}")
        difference_columns.append(col)
    print("\n"+("==="*30))
    print("\n")

Cantidad de columnas con diferencias

In [ ]:
print(f"Columns with differences: {len(difference_columns)}/{len(columns)} \n")
print(difference_columns)
for col in difference_columns:
    print("\n"+("==="*30))
    print(f"geopackage {col} dtype: {geopackage.dtypes[col]}")
    print(f"geopackage2 {col} dtype: {geopackage2.dtypes[col]}")

Comparacion de datos individuales entre los 2 GeoDataFrames

In [ ]:
geopackage_cols = geopackage.columns
differences = {}
for col in geopackage.columns:
    col_len = len(geopackage[col])
    geopackage_differences = []
    geopackage2_differences = []
    for data in range(col_len):
        if geopackage[col][data] != geopackage2[col][data]:
            #print(f"Data {data} in column {col} is not Equal on both Geodataframes")
            geopackage_differences.append(geopackage[col][data])
            geopackage2_differences.append(geopackage2[col][data])
    if len(geopackage_differences) > 0:
        print(f"differences on Col: {col}")
        for data in range(len(geopackage_differences)):
            data1 = geopackage_differences[data]
            data2 = geopackage2_differences[data]
            print(f"{data1}\t<===>\t{data2}")
            print(f"{count_decimals(data1)}\t<===>\t{count_decimals(data2)}\n")
        print("\n")